In [15]:
import os
from pathlib import Path
import json
from jsonargparse import CLI
import boto3

import time
from copy import deepcopy
import threading
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TextGenerationPipeline, AutoModelForSequenceClassification
from peft import PeftModel
from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest
from accelerate import Accelerator
import multiprocessing as mp
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed

def get_batch_response(texts, base_model, model_path, temperature, max_tokens, top_p):

    model = LLM(model=base_model,
                enable_lora=True,
                tensor_parallel_size=8,
                dtype="bfloat16",
                max_lora_rank=64)

    lora_req = LoRARequest("lora1",1,model_path)
    sampling_params = SamplingParams(max_tokens=max_tokens, temperature=temperature, top_p=top_p)

    results = model.generate(texts, sampling_params, lora_request=lora_req)

    completions = [o.outputs[0].text for o in results]

    return completions


def main(from_json: str = None, to_json: str = None, prompt: str = None, base_model: str = 'llama-3.1-instruct',
         model_path: str = 'llama-3.1-instruct', temperature: float = 0, max_tokens: int = 512, top_p=0.9,
         n_samples: int = -1, input_field: str = 'input', existing_json: str = None):
    EXSTING = {}
    if existing_json is not None:
        with open(existing_json, 'r') as f:
            for l in f.readlines():
                d = json.loads(l)
                if d['resp'] != 'API Failed':
                    EXSTING[d['prompt']] = d
    
    
    path = Path(to_json)
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        path.touch()

    with open(from_json, "r") as fr, open(to_json, 'w') as fw:

        lines = fr.readlines()
        total_lines = min(len(lines), n_samples) if n_samples > 0 else len(lines)
        start_time = time.time()
           
        texts = [prompt.format(json.loads(lines[i])[input_field]) for i in range(len(lines))]
        
        results = get_batch_response(
                            texts, base_model, model_path, temperature, max_tokens, top_p
                        )
        
        for result in results:
            fw.write(json.dumps(result) + '\n')


dataset = "inspired"
model = "llama3-2-1b-instruct"
local_folder = "test_epoch5_seed2_conv_entropy_lr2e-5"
alg = 'sft_generated'

main(from_json='testsets/'+dataset+'/test.jsonl',
    to_json=f'test_res/{alg}/{dataset}/{model}/{dataset}_test_conv_entropy_ep5lr2e-5.jsonl',
    prompt="Pretend you are a movie recommender system. I will give you a conversation between a user and you (a recommender system). Based on the conversation, reply 5 recommendations in the format of '1. [Movie Name]\n 2. [Movie Name]\n 3. [Movie Name]\n'. Then terminate the conversation. Here is the conversation: {}",
    base_model='meta-llama/Llama-3.2-1B-Instruct',
    model_path=f'../../outputs/{alg}/{dataset}/{model}/{local_folder}',
    temperature=0.1,
    max_tokens=512,
    n_samples=-1)

INFO 08-04 02:25:47 [config.py:823] This model supports multiple tasks: {'reward', 'classify', 'generate', 'embed', 'score'}. Defaulting to 'generate'.
INFO 08-04 02:25:47 [config.py:1946] Defaulting to use mp for distributed inference
INFO 08-04 02:25:47 [config.py:2195] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 08-04 02:25:47 [core.py:455] Waiting for init message from front-end.
INFO 08-04 02:25:47 [core.py:70] Initializing a V1 LLM engine (v0.9.1) with config: model='meta-llama/Llama-3.2-1B-Instruct', speculative_config=None, tokenizer='meta-llama/Llama-3.2-1B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config={}, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=131072, download_dir=None, load_format=auto, tensor_parallel_size=8, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto,  device_config=cuda, decoding_c

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


WARNING 08-04 02:25:48 [utils.py:2737] Methods determine_num_available_blocks,device_config,get_cache_block_size_bytes,initialize_cache not implemented in <vllm.v1.worker.gpu_worker.Worker object at 0x7f2b695def50>
(VllmWorker rank=0 pid=37425) INFO 08-04 02:25:48 [shm_broadcast.py:289] vLLM message queue communication handle: Handle(local_reader_ranks=[0], buffer_handle=(1, 10485760, 10, 'psm_c32dcfa9'), local_subscribe_addr='ipc:///tmp/0fe2a669-cbf5-481d-95fa-ab6d6a8d089a', remote_subscribe_addr=None, remote_addr_ipv6=False)
WARNING 08-04 02:25:48 [utils.py:2737] Methods determine_num_available_blocks,device_config,get_cache_block_size_bytes,initialize_cache not implemented in <vllm.v1.worker.gpu_worker.Worker object at 0x7f2b695dc670>
(VllmWorker rank=1 pid=37426) INFO 08-04 02:25:48 [shm_broadcast.py:289] vLLM message queue communication handle: Handle(local_reader_ranks=[0], buffer_handle=(1, 10485760, 10, 'psm_a73af705'), local_subscribe_addr='ipc:///tmp/75949bf0-b11c-4d2e-8bfb-2

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


(VllmWorker rank=1 pid=37426) INFO 08-04 02:25:54 [punica_selector.py:19] Using PunicaWrapperGPU.
(VllmWorker rank=2 pid=37427) INFO 08-04 02:25:54 [default_loader.py:272] Loading weights took 0.10 seconds
(VllmWorker rank=2 pid=37427) INFO 08-04 02:25:54 [punica_selector.py:19] Using PunicaWrapperGPU.
(VllmWorker rank=0 pid=37425) INFO 08-04 02:25:54 [default_loader.py:272] Loading weights took 0.13 seconds
(VllmWorker rank=4 pid=37429) INFO 08-04 02:25:54 [gpu_model_runner.py:1624] Model loading took 0.3804 GiB and 0.478797 seconds
(VllmWorker rank=0 pid=37425) INFO 08-04 02:25:54 [punica_selector.py:19] Using PunicaWrapperGPU.
(VllmWorker rank=5 pid=37430) INFO 08-04 02:25:54 [gpu_model_runner.py:1624] Model loading took 0.3804 GiB and 0.526649 seconds
(VllmWorker rank=6 pid=37431) INFO 08-04 02:25:54 [gpu_model_runner.py:1624] Model loading took 0.3804 GiB and 0.587828 seconds
(VllmWorker rank=3 pid=37428) INFO 08-04 02:25:54 [gpu_model_runner.py:1624] Model loading took 0.3804 GiB

Adding requests:   0%|          | 0/228 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/228 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]